# A1.4 · Blast radius as a design metric

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

Builds on **[A1.3 · Authorization models that make bad grants impossible](https://spbreed.github.io/cyber-commons/lessons/A1.3.html)**.

| | |
|---|---|
| Open-source tooling | OpenFGA, SPIRE |
| Open-weight models | Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


"Reduce the blast radius" is advice. Advice does not survive a roadmap
discussion, because it cannot be traded off against a delivery date.

A **number** survives. It moves when you change the design, you can put it in a
review, and — most usefully — it can go into CI and fail a build.

The metric used throughout this curriculum is deliberately crude:

    blast radius = Σ over state-changing tools of
                     scope_weight × (2 if irreversible) × (0 if gated)

The absolute value means nothing. The **ratio between two designs** means a
great deal, and that is all a design metric has to do. Anyone who demands a
calibrated number before measuring anything ends up measuring nothing.

## 2 · Demo — four ways to build the same capability

The requirement: an agent that can triage security findings, fix simple ones, and deploy the fix. Four architectures, all of which deliver it.

In [ ]:
from dataclasses import dataclass

SCOPE_WEIGHT = {"self": 1, "project": 3, "tenant": 8, "org": 20, "internet": 50}

@dataclass(frozen=True)
class Tool:
    name: str; writes: bool = False; reversible: bool = True; scope: str = "self"

def blast(tools, gated=frozenset()):
    total = 0
    for t in tools:
        if not t.writes or t.name in gated:
            continue
        total += SCOPE_WEIGHT[t.scope] * (1 if t.reversible else 2)
    return total

READ   = [Tool("read_findings"), Tool("read_source")]
FIX    = [Tool("write_file", writes=True, scope="project"),
          Tool("open_pr",    writes=True, scope="project")]
SHIP   = [Tool("merge_pr", writes=True, scope="project", reversible=False),
          Tool("deploy",   writes=True, scope="org",     reversible=False)]

designs = {
    "A · one agent, everything":        (READ + FIX + SHIP, set()),
    "B · one agent, gate the shipping": (READ + FIX + SHIP, {"merge_pr", "deploy"}),
    "C · two agents (fixer / shipper)": (READ + FIX,        set()),
    "D · two agents + gated shipper":   (READ + FIX,        set()),
}
for name, (tools, gated) in designs.items():
    print(f"{name:36s} blast = {blast(tools, gated):3d}")

print("\nDesign D's shipper agent, measured separately:")
print(f"{'    shipper (gated)':36s} blast = {blast(SHIP, {'merge_pr','deploy'}):3d}")
print(f"{'    shipper (ungated)':36s} blast = {blast(SHIP):3d}   ← the honest number")

## 3 · Where it breaks — the number can lie

Design C looks best: blast 6. But it achieved that by *moving* the dangerous tools to another agent, not by removing them. If you measure each agent separately and report the lowest, you have optimised the metric rather than the risk. This is Goodhart's law arriving on schedule.

The metric is only honest when it is computed **over the whole system**, including every agent that can be reached from the first one.

In [ ]:
def system_blast(agents):
    """Sum across every agent in the system, not the one you are reviewing."""
    return sum(blast(tools, gated) for tools, gated in agents.values())

split_honest = {
    "fixer":   (READ + FIX, set()),
    "shipper": (SHIP,       set()),          # someone still runs this
}
split_gated = {
    "fixer":   (READ + FIX, set()),
    "shipper": (SHIP,       {"merge_pr", "deploy"}),
}
mono = {"one-agent": (READ + FIX + SHIP, set())}

for name, agents in (("monolith", mono), ("split, shipper ungated", split_honest),
                     ("split, shipper gated", split_gated)):
    print(f"{name:26s} system blast = {system_blast(agents):3d}")
print("\nSplitting alone bought nothing. Splitting AND gating bought everything.")
print("Reporting only the fixer's number would have hidden that.")

## 4 · The control — put it in CI

A metric nobody computes is a metric nobody has. The version that works is a budget, enforced by the build.

In [ ]:
BUDGETS = {"L1": 0, "L2": 0, "L2.5": 20, "L3": 60}

def check_budget(system, rung):
    total = system_blast(system)
    budget = BUDGETS[rung]
    ok = total <= budget
    return ok, (f"system blast {total} {'≤' if ok else '>'} budget {budget} "
                f"for rung {rung}")

for name, system, rung in [
    ("split + gated shipper", split_gated,  "L2.5"),
    ("split, ungated shipper", split_honest, "L2.5"),
    ("monolith",              mono,         "L2.5"),
]:
    ok, msg = check_budget(system, rung)
    print(f"{'PASS' if ok else 'FAIL'}  {name:26s} {msg}")

ok, _ = check_budget(split_gated, "L2.5")
assert ok, "the intended design must pass its own budget"
print("\nWired into CI, adding a tool now fails the build unless someone either")
print("gates it or raises the budget deliberately — which is a decision with a name on it.")

## What you just proved

Design A scores 92, B scores 6, C scores 6. Measured across the whole system, the ungated split still scores 92 while the gated split scores 6 — showing that splitting alone bought nothing. The budget check passes only the gated design.

## Your turn

Compute the system blast radius for your largest agent deployment, counting every agent it can invoke. Then pick a budget and see how many of your current designs would fail it. Set the budget at today's number and ratchet down; a budget nothing passes is ignored by lunchtime.

---

**Next → [A1.5 · Multi-agent topology](https://spbreed.github.io/cyber-commons/lessons/A1.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*